In [ ]:
# Example 1: Basic LoRA (Low-Rank Adaptation) Configuration
# Demonstrate LoRA configuration for parameter-efficient fine-tuning

import peft
from peft import LoraConfig, get_peft_model, TaskType
import torch
import torch.nn as nn

def create_simple_model():
    """Create a simple model for LoRA demonstration"""
    class SimpleTransformer(nn.Module):
        def __init__(self, vocab_size=1000, hidden_size=512, num_layers=2):
            super().__init__()
            self.embedding = nn.Embedding(vocab_size, hidden_size)
            self.layers = nn.ModuleList([
                nn.TransformerEncoderLayer(
                    d_model=hidden_size,
                    nhead=8,
                    dim_feedforward=2048,
                    batch_first=True
                )
                for _ in range(num_layers)
            ])
            self.output = nn.Linear(hidden_size, vocab_size)
            
        def forward(self, x):
            x = self.embedding(x)
            for layer in self.layers:
                x = layer(x)
            return self.output(x)
    
    return SimpleTransformer()

def demonstrate_lora_configuration():
    """Show LoRA configuration and model adaptation"""
    
    print("=== LoRA Configuration Demo ===")
    
    # Create base model
    base_model = create_simple_model()
    print(f"Base model parameters: {sum(p.numel() for p in base_model.parameters()):,}")
    
    # Configure LoRA. A custom nn.Module takes NO task_type: the CAUSAL_LM wrapper needs a
    # transformers-style model (prepare_inputs_for_generation, generate). nn.TransformerEncoderLayer's
    # nn.Linear layers are linear1/linear2 (its q/k/v projection is one fused MultiheadAttention weight).
    lora_config = LoraConfig(
        r=16,  # Rank of adaptation
        lora_alpha=32,  # LoRA scaling parameter
        target_modules=["linear1", "linear2"],  # Target linear layers
        lora_dropout=0.1,
        bias="none",
    )
    
    print(f"\nLoRA Configuration:")
    print(f"  Rank (r): {lora_config.r}")
    print(f"  Alpha: {lora_config.lora_alpha}")
    print(f"  Target modules: {lora_config.target_modules}")
    print(f"  Dropout: {lora_config.lora_dropout}")
    print(f"  Task type: {lora_config.task_type}")
    
    # Apply LoRA to model
    original_params = sum(p.numel() for p in base_model.parameters())
    peft_model = get_peft_model(base_model, lora_config)
    
    # Count trainable parameters
    total_params = sum(p.numel() for p in peft_model.parameters())
    trainable_params = sum(p.numel() for p in peft_model.parameters() if p.requires_grad)
    
    print(f"\nAfter applying LoRA:")
    print(f"  Total parameters: {total_params:,}")
    print(f"  Trainable parameters: {trainable_params:,}")
    print(f"  Trainable %: {100 * trainable_params / total_params:.2f}%")
    
    # Print trainable parameters info
    peft_model.print_trainable_parameters()
    
    # Each wrapped nn.Linear(in, out) gains A (r x in) + B (out x r); only those train.
    wrapped = [m.base_layer for m in peft_model.modules() if isinstance(m, peft.tuners.lora.LoraLayer)]
    assert len(wrapped) == 4, wrapped  # linear1 + linear2 in each of the 2 encoder layers
    assert trainable_params == sum(lora_config.r * (m.in_features + m.out_features) for m in wrapped) == 163_840
    assert total_params == original_params + trainable_params
    assert peft_model(torch.randint(0, 1000, (2, 8))).shape == (2, 8, 1000)
    
    return lora_config

def demonstrate_peft_types():
    """Show different PEFT method types and configurations"""
    
    print(f"\n=== PEFT Method Types ===")
    
    # LoRA configuration
    lora_config = LoraConfig(r=8, lora_alpha=16, lora_dropout=0.1)
    print(f"LoRA Config: {lora_config}")
    
    # Different task types
    task_types = [
        TaskType.SEQ_CLS,
        TaskType.TOKEN_CLS, 
        TaskType.CAUSAL_LM,
        TaskType.SEQ_2_SEQ_LM
    ]
    
    print(f"\nAvailable Task Types:")
    for task_type in task_types:
        print(f"  {task_type.value}: {task_type}")
    
    # PEFT method comparison
    methods_info = {
        "LoRA": "Low-Rank Adaptation - adds trainable rank decomposition matrices",
        "AdaLoRA": "Adaptive LoRA - dynamically adjusts rank during training",  
        "Prefix Tuning": "Prepends trainable prefix tokens to input",
        "P-Tuning": "Prompt tuning with trainable continuous prompts",
        "IA3": "Infused Adapter by Inhibiting and Amplifying Inner Activations"
    }
    
    print(f"\nPEFT Methods Overview:")
    for method, description in methods_info.items():
        print(f"  {method}: {description}")

# Run LoRA demonstrations
print("=== PEFT (Parameter-Efficient Fine-Tuning) Basics ===")

lora_config = demonstrate_lora_configuration()
demonstrate_peft_types()

print("Example 1: Basic LoRA configuration completed successfully!")

In [ ]:
# Example 2: Advanced PEFT Methods and Configurations
# Demonstrate different PEFT methods: AdaLoRA, Prefix Tuning, and IA3

import peft
from peft import (
    LoraConfig, AdaLoraConfig, PrefixTuningConfig, 
    PromptTuningConfig, IA3Config, TaskType
)
import torch
import torch.nn as nn

def demonstrate_adalora_config():
    """Show AdaLoRA (Adaptive LoRA) configuration"""
    
    print("=== AdaLoRA Configuration ===")
    
    adalora_config = AdaLoraConfig(
        peft_type="ADALORA",
        task_type=TaskType.CAUSAL_LM,
        r=12,  # Initial rank
        lora_alpha=32,
        target_modules=["q_proj", "k_proj", "v_proj"],
        lora_dropout=0.1,
        # AdaLoRA specific parameters
        target_r=8,  # Target rank after pruning
        init_r=12,   # Initial rank
        tinit=200,   # Steps before starting rank reduction
        tfinal=1000, # Steps to finish rank reduction
        deltaT=10,   # Steps between rank updates
        beta1=0.85,  # Exponential moving average parameter
        beta2=0.85,  # Exponential moving average parameter
        orth_reg_weight=0.5,  # Orthogonal regularization weight
        total_step=2000,  # Total training steps (required for AdaLoRA)
    )
    
    print(f"AdaLoRA Configuration:")
    print(f"  Initial rank: {adalora_config.init_r}")
    print(f"  Target rank: {adalora_config.target_r}")
    print(f"  Total steps: {adalora_config.total_step}")
    print(f"  Rank reduction schedule: {adalora_config.tinit} -> {adalora_config.tfinal} steps")
    print(f"  Update interval: {adalora_config.deltaT} steps")
    print(f"  Orthogonal regularization: {adalora_config.orth_reg_weight}")
    
    return adalora_config

def demonstrate_prefix_tuning_config():
    """Show Prefix Tuning configuration"""
    
    print(f"\n=== Prefix Tuning Configuration ===")
    
    prefix_config = PrefixTuningConfig(
        task_type=TaskType.CAUSAL_LM,
        num_virtual_tokens=20,  # Number of prefix tokens
        token_dim=512,          # Token dimension
        num_transformer_submodules=2,  # Number of transformer layers
        num_attention_heads=8,
        num_layers=12,
        encoder_hidden_size=512,
        prefix_projection=True,  # Use projection for prefix
    )
    
    print(f"Prefix Tuning Configuration:")
    print(f"  Virtual tokens: {prefix_config.num_virtual_tokens}")
    print(f"  Token dimension: {prefix_config.token_dim}")
    print(f"  Attention heads: {prefix_config.num_attention_heads}")
    print(f"  Layers: {prefix_config.num_layers}")
    print(f"  Use projection: {prefix_config.prefix_projection}")
    
    # Estimate parameter count for prefix tuning
    prefix_params = (prefix_config.num_virtual_tokens * 
                    prefix_config.token_dim * 
                    prefix_config.num_layers * 2)  # Key and value
    print(f"  Estimated prefix parameters: {prefix_params:,}")
    
    return prefix_config

def demonstrate_prompt_tuning_config():
    """Show Prompt Tuning configuration"""
    
    print(f"\n=== Prompt Tuning Configuration ===")
    
    prompt_config = PromptTuningConfig(
        task_type=TaskType.CAUSAL_LM,
        num_virtual_tokens=50,   # Number of soft prompt tokens
        token_dim=512,           # Hidden size
        prompt_tuning_init="RANDOM",  # Use random initialization
    )
    
    print(f"Prompt Tuning Configuration:")
    print(f"  Virtual tokens: {prompt_config.num_virtual_tokens}")
    print(f"  Token dimension: {prompt_config.token_dim}")
    print(f"  Initialization: {prompt_config.prompt_tuning_init}")
    
    # Estimate parameters for prompt tuning
    prompt_params = prompt_config.num_virtual_tokens * prompt_config.token_dim
    print(f"  Prompt parameters: {prompt_params:,}")
    
    return prompt_config

def demonstrate_ia3_config():
    """Show IA3 (Infused Adapter by Inhibiting and Amplifying Inner Activations) configuration"""
    
    print(f"\n=== IA3 Configuration ===")
    
    ia3_config = IA3Config(
        task_type=TaskType.CAUSAL_LM,
        target_modules=["k_proj", "v_proj", "down_proj"],  # Key, value, and feedforward projections
        feedforward_modules=["down_proj"],  # Which modules are feedforward
    )
    
    print(f"IA3 Configuration:")
    print(f"  Target modules: {ia3_config.target_modules}")
    print(f"  Feedforward modules: {ia3_config.feedforward_modules}")
    print(f"  Task type: {ia3_config.task_type}")
    
    # IA3 adds learned vectors that scale activations
    # Very parameter efficient - only adds vectors equal to hidden dimensions
    print(f"  Parameter efficiency: Adds only scaling vectors (very efficient)")
    
    return ia3_config

def compare_peft_methods():
    """Compare different PEFT methods"""
    
    print(f"\n=== PEFT Method Comparison ===")
    
    # Mock model dimensions for comparison
    hidden_size = 4096
    num_layers = 32
    vocab_size = 32000
    
    methods_comparison = {
        "LoRA (r=16)": {
            "params": 16 * 2 * hidden_size * num_layers,  # Approximate
            "description": "Low-rank matrices added to linear layers",
            "efficiency": "Medium",
            "performance": "High"
        },
        "AdaLoRA (r=8-16)": {
            "params": 12 * 2 * hidden_size * num_layers,  # Variable rank
            "description": "Adaptive rank reduction during training",
            "efficiency": "High", 
            "performance": "High"
        },
        "Prefix Tuning (20 tokens)": {
            "params": 20 * hidden_size * num_layers * 2,
            "description": "Trainable prefix tokens prepended to inputs",
            "efficiency": "Medium",
            "performance": "Medium"
        },
        "Prompt Tuning (50 tokens)": {
            "params": 50 * hidden_size,
            "description": "Soft prompt tokens in embedding space",
            "efficiency": "Very High",
            "performance": "Medium"
        },
        "IA3": {
            "params": hidden_size * num_layers * 3,  # k, v, ffn scaling
            "description": "Learned scaling vectors for activations",
            "efficiency": "Very High",
            "performance": "Medium"
        }
    }
    
    print(f"Method comparison (for {hidden_size}d model with {num_layers} layers):")
    for method, stats in methods_comparison.items():
        print(f"  {method}:")
        print(f"    Parameters: ~{stats['params']:,}")
        print(f"    Description: {stats['description']}")
        print(f"    Efficiency: {stats['efficiency']}")
        print(f"    Performance: {stats['performance']}")
        print()

# Run advanced PEFT method demonstrations
print("=== PEFT Advanced Methods and Configurations ===")

adalora_config = demonstrate_adalora_config()
prefix_config = demonstrate_prefix_tuning_config()
prompt_config = demonstrate_prompt_tuning_config()
ia3_config = demonstrate_ia3_config()
compare_peft_methods()

# The configs keep the values they were given
from peft import PeftType
assert adalora_config.peft_type == PeftType.ADALORA and adalora_config.task_type == TaskType.CAUSAL_LM
assert (adalora_config.r, adalora_config.lora_alpha) == (12, 32)
assert set(adalora_config.target_modules) == {"q_proj", "k_proj", "v_proj"}
assert (adalora_config.init_r, adalora_config.target_r, adalora_config.total_step) == (12, 8, 2000)
assert (adalora_config.tinit, adalora_config.tfinal, adalora_config.deltaT) == (200, 1000, 10)

assert prefix_config.peft_type == PeftType.PREFIX_TUNING
assert (prefix_config.num_virtual_tokens, prefix_config.token_dim, prefix_config.num_layers) == (20, 512, 12)
assert prefix_config.prefix_projection is True and prefix_config.encoder_hidden_size == 512

assert prompt_config.peft_type == PeftType.PROMPT_TUNING
assert (prompt_config.num_virtual_tokens, prompt_config.token_dim) == (50, 512)
assert prompt_config.prompt_tuning_init == "RANDOM"

assert ia3_config.peft_type == PeftType.IA3 and ia3_config.task_type == TaskType.CAUSAL_LM
assert set(ia3_config.target_modules) == {"k_proj", "v_proj", "down_proj"}
assert set(ia3_config.feedforward_modules) == {"down_proj"}

# A config survives a round trip through its dict form
from peft import get_peft_config
rebuilt = get_peft_config(adalora_config.to_dict())
assert isinstance(rebuilt, AdaLoraConfig) and (rebuilt.init_r, rebuilt.target_r) == (12, 8)

print("\nExample 2: Advanced PEFT methods completed successfully!")

In [ ]:
# Example 3: PEFT Model Training and Fine-tuning Workflow
# Demonstrate complete PEFT training workflow and model management

import peft
from peft import LoraConfig, get_peft_model, TaskType, PeftModel
import torch
import torch.nn as nn
import torch.optim as optim

class MockTransformerModel(nn.Module):
    """Mock transformer model for PEFT demonstration"""
    
    def __init__(self, vocab_size=1000, hidden_size=512, num_layers=6):
        super().__init__()
        self.vocab_size = vocab_size
        self.hidden_size = hidden_size
        self.embedding = nn.Embedding(vocab_size, hidden_size)
        self.position_embedding = nn.Embedding(1024, hidden_size)
        
        # Create a config object for PEFT compatibility (dict-like)
        class ModelConfig:
            def __init__(self, vocab_size, hidden_size):
                self.vocab_size = vocab_size
                self.hidden_size = hidden_size
                self.model_type = "custom_transformer"
            
            def get(self, key, default=None):
                """Make config dict-like for PEFT compatibility"""
                return getattr(self, key, default)

            def to_dict(self):
                """PEFT reads the model config as a dict when it saves an adapter"""
                return dict(vars(self))
        
        self.config = ModelConfig(vocab_size, hidden_size)
        
        # Create layers with the module names that PEFT expects
        self.layers = nn.ModuleList()
        for i in range(num_layers):
            layer = nn.ModuleDict({
                'self_attn': nn.ModuleDict({
                    'q_proj': nn.Linear(hidden_size, hidden_size),
                    'k_proj': nn.Linear(hidden_size, hidden_size), 
                    'v_proj': nn.Linear(hidden_size, hidden_size),
                    'out_proj': nn.Linear(hidden_size, hidden_size),
                }),
                'mlp': nn.ModuleDict({
                    'up_proj': nn.Linear(hidden_size, hidden_size * 4),
                    'down_proj': nn.Linear(hidden_size * 4, hidden_size),
                }),
                'layer_norm1': nn.LayerNorm(hidden_size),
                'layer_norm2': nn.LayerNorm(hidden_size),
            })
            self.layers.append(layer)
        
        self.final_layer_norm = nn.LayerNorm(hidden_size)
        self.output_projection = nn.Linear(hidden_size, vocab_size)
    
    def forward(self, input_ids=None, attention_mask=None, inputs_embeds=None, **kwargs):
        # Handle both input_ids and inputs_embeds (PEFT compatibility)
        if inputs_embeds is None:
            if input_ids is None:
                raise ValueError("Either input_ids or inputs_embeds must be provided")
            seq_len = input_ids.size(1)
            positions = torch.arange(seq_len, device=input_ids.device)
            x = self.embedding(input_ids) + self.position_embedding(positions)
        else:
            seq_len = inputs_embeds.size(1)
            positions = torch.arange(seq_len, device=inputs_embeds.device)
            x = inputs_embeds + self.position_embedding(positions)
        
        for layer in self.layers:
            # Self-attention
            q = layer['self_attn']['q_proj'](x)
            k = layer['self_attn']['k_proj'](x) 
            v = layer['self_attn']['v_proj'](x)
            
            # Simplified attention (no masking for demo)
            attn = torch.softmax(q @ k.transpose(-2, -1) / (x.size(-1) ** 0.5), dim=-1)
            attn_out = layer['self_attn']['out_proj'](attn @ v)
            x = layer['layer_norm1'](x + attn_out)
            
            # MLP
            mlp_out = layer['mlp']['down_proj'](torch.relu(layer['mlp']['up_proj'](x)))
            x = layer['layer_norm2'](x + mlp_out)
        
        x = self.final_layer_norm(x)
        return self.output_projection(x)
    
    def prepare_inputs_for_generation(self, input_ids, **kwargs):
        """Required method for PEFT compatibility"""
        return {"input_ids": input_ids}

def demonstrate_peft_training_setup():
    """Show complete PEFT training setup"""
    
    print("=== PEFT Training Setup ===")
    
    # Create base model (seeded, so the same base weights can be rebuilt later)
    torch.manual_seed(0)
    base_model = MockTransformerModel(vocab_size=1000, hidden_size=256, num_layers=4)
    original_params = sum(p.numel() for p in base_model.parameters())
    print(f"Base model parameters: {original_params:,}")
    
    # Configure LoRA
    lora_config = LoraConfig(
        r=8,
        lora_alpha=16,
        target_modules=["q_proj", "k_proj", "v_proj", "out_proj"],
        lora_dropout=0.1,
        bias="none",
        task_type=TaskType.CAUSAL_LM,
    )
    
    # Apply PEFT to model
    peft_model = get_peft_model(base_model, lora_config)
    
    # Get parameter statistics
    total_params = sum(p.numel() for p in peft_model.parameters())
    trainable_params = sum(p.numel() for p in peft_model.parameters() if p.requires_grad)
    
    print(f"After PEFT application:")
    print(f"  Total parameters: {total_params:,}")
    print(f"  Trainable parameters: {trainable_params:,}")
    print(f"  Percentage trainable: {100 * trainable_params / total_params:.2f}%")
    
    # Show trainable parameters breakdown
    peft_model.print_trainable_parameters()
    
    return peft_model, lora_config

def demonstrate_peft_training_loop():
    """Show PEFT training loop simulation"""
    
    print(f"\n=== PEFT Training Loop Simulation ===")
    
    # Get PEFT model
    peft_model, config = demonstrate_peft_training_setup()
    
    # Setup optimizer (only for PEFT parameters)
    optimizer = optim.AdamW(
        [p for p in peft_model.parameters() if p.requires_grad],
        lr=1e-4,
        weight_decay=0.01
    )
    
    # Setup loss function
    loss_fn = nn.CrossEntropyLoss()
    
    # Simulate training data
    batch_size, seq_len = 4, 32
    vocab_size = 1000
    
    print(f"Training configuration:")
    print(f"  Optimizer: AdamW (lr=1e-4, wd=0.01)")
    print(f"  Batch size: {batch_size}")
    print(f"  Sequence length: {seq_len}")
    print(f"  Trainable params: {sum(p.numel() for p in peft_model.parameters() if p.requires_grad):,}")
    
    # Training simulation
    peft_model.train()
    training_losses = []
    
    for epoch in range(3):
        epoch_losses = []
        
        for step in range(5):  # 5 steps per epoch for demo
            # Generate random training data
            input_ids = torch.randint(0, vocab_size, (batch_size, seq_len))
            labels = torch.randint(0, vocab_size, (batch_size, seq_len))
            
            # Forward pass
            outputs = peft_model(input_ids)
            loss = loss_fn(outputs.view(-1, vocab_size), labels.view(-1))
            
            # Backward pass
            optimizer.zero_grad()
            loss.backward()
            
            # Gradient clipping
            torch.nn.utils.clip_grad_norm_(peft_model.parameters(), max_norm=1.0)
            
            optimizer.step()
            
            epoch_losses.append(loss.item())
        
        avg_loss = sum(epoch_losses) / len(epoch_losses)
        training_losses.append(avg_loss)
        print(f"Epoch {epoch + 1}: Average loss = {avg_loss:.4f}")
    
    # Training summary
    print(f"\nTraining completed:")
    print(f"  Initial loss: {training_losses[0]:.4f}")
    print(f"  Final loss: {training_losses[-1]:.4f}")
    print(f"  Loss reduction: {((training_losses[0] - training_losses[-1]) / training_losses[0] * 100):.1f}%")
    
    return peft_model

def demonstrate_peft_model_management():
    """Show PEFT model saving/loading and merging"""
    
    print(f"\n=== PEFT Model Management ===")
    
    # Train a PEFT model (simplified)
    peft_model = demonstrate_peft_training_loop()
    
    # Model state management
    print(f"Model management operations:")
    
    # The adapter configuration as a dict
    adapter_state = peft_model.peft_config["default"].to_dict()
    print(f"  Adapter config keys: {list(adapter_state.keys())}")

    # Round-trip through get_peft_config: dict -> concrete LoraConfig
    from peft import get_peft_config
    rebuilt_config = get_peft_config(dict(adapter_state))
    assert type(rebuilt_config).__name__ == "LoraConfig" and rebuilt_config.r == 8
    print(f"  get_peft_config round-trip: {type(rebuilt_config).__name__}(r={rebuilt_config.r})")
    
    # Show model structure
    print(f"  Active adapters: {list(peft_model.peft_config.keys())}")
    print(f"  Base model type: {type(peft_model.get_base_model()).__name__}")
    assert list(peft_model.peft_config) == ["default"]
    assert isinstance(peft_model.get_base_model(), MockTransformerModel)

    # Save the adapter, then load it into a FRESH base model and compare
    import os
    import tempfile
    peft_model.eval()  # no LoRA dropout, so outputs are comparable
    test_input = torch.randint(0, 1000, (1, 10))
    with torch.no_grad():
        test_output = peft_model(test_input)
    print(f"  Inference test: Input shape {test_input.shape} -> Output shape {test_output.shape}")
    assert test_output.shape == (1, 10, 1000)

    adapter_weights = {name: p.detach().clone() for name, p in peft_model.named_parameters() if "lora_" in name}
    with tempfile.TemporaryDirectory() as adapter_dir:
        peft_model.save_pretrained(adapter_dir)
        saved_files = sorted(os.listdir(adapter_dir))
        print(f"  Saved adapter files: {saved_files}")
        assert "adapter_config.json" in saved_files and "adapter_model.safetensors" in saved_files

        # Rebuild the base model from the same seed: its weights match the frozen base weights
        # of the trained model, and the adapter weights come only from the saved files
        torch.manual_seed(0)
        fresh_base = MockTransformerModel(vocab_size=1000, hidden_size=256, num_layers=4)
        reloaded = PeftModel.from_pretrained(fresh_base, adapter_dir)

    reloaded_weights = {name: p.detach() for name, p in reloaded.named_parameters() if "lora_" in name}
    assert reloaded_weights.keys() == adapter_weights.keys() and len(adapter_weights) == 32
    assert all(torch.equal(reloaded_weights[name], adapter_weights[name]) for name in adapter_weights)
    reloaded.eval()
    with torch.no_grad():
        assert torch.allclose(reloaded(test_input), test_output)
    print(f"  Reloaded adapter: {len(reloaded_weights)} LoRA tensors identical to the saved ones")

    # Adapter operations available on the model
    for method in ("save_pretrained", "load_adapter", "set_adapter", "disable_adapter", "merge_and_unload"):
        assert callable(getattr(peft_model, method, None)), method
    print(f"  Adapter operations present: save_pretrained, load_adapter, set_adapter, disable_adapter, merge_and_unload")
    
    return peft_model

# Run PEFT training demonstrations
print("=== PEFT Training and Fine-tuning Workflow ===")

trained_model = demonstrate_peft_model_management()

print("\nExample 3: PEFT training workflow completed successfully!")

In [ ]:
# Example 4: Multi-Adapter Management and Task-Specific Fine-tuning
# Demonstrate multiple adapter management, task switching, and adapter composition

import peft
from peft import LoraConfig, get_peft_model, TaskType
import torch
import torch.nn as nn

def create_base_model_for_multi_task():
    """Create a base model suitable for multiple tasks"""
    class MultiTaskTransformer(nn.Module):
        def __init__(self, vocab_size=1000, hidden_size=512, num_layers=4):
            super().__init__()
            self.vocab_size = vocab_size
            self.hidden_size = hidden_size
            self.embedding = nn.Embedding(vocab_size, hidden_size)
            
            # Create a config object for PEFT compatibility (dict-like)
            class ModelConfig:
                def __init__(self, vocab_size, hidden_size):
                    self.vocab_size = vocab_size
                    self.hidden_size = hidden_size
                    self.model_type = "custom_multi_task_transformer"
                
                def get(self, key, default=None):
                    """Make config dict-like for PEFT compatibility"""
                    return getattr(self, key, default)
            
            self.config = ModelConfig(vocab_size, hidden_size)
            
            self.transformer_layers = nn.ModuleList([
                nn.ModuleDict({
                    'attention': nn.ModuleDict({
                        'q_proj': nn.Linear(hidden_size, hidden_size),
                        'k_proj': nn.Linear(hidden_size, hidden_size),
                        'v_proj': nn.Linear(hidden_size, hidden_size),
                        'out_proj': nn.Linear(hidden_size, hidden_size),
                    }),
                    'feed_forward': nn.ModuleDict({
                        'up_proj': nn.Linear(hidden_size, hidden_size * 2),
                        'down_proj': nn.Linear(hidden_size * 2, hidden_size),
                    }),
                    'norm1': nn.LayerNorm(hidden_size),
                    'norm2': nn.LayerNorm(hidden_size),
                })
                for _ in range(num_layers)
            ])
            
            # Multiple task heads
            self.classification_head = nn.Linear(hidden_size, 10)  # 10 classes
            self.generation_head = nn.Linear(hidden_size, vocab_size)
            self.sequence_head = nn.Linear(hidden_size, 2)  # Binary sequence classification
            
        def forward(self, input_ids=None, task_type="generation", attention_mask=None, inputs_embeds=None, **kwargs):
            # Handle both input_ids and inputs_embeds (PEFT compatibility)
            if inputs_embeds is None:
                if input_ids is None:
                    raise ValueError("Either input_ids or inputs_embeds must be provided")
                x = self.embedding(input_ids)
            else:
                x = inputs_embeds
            
            for layer in self.transformer_layers:
                # Attention
                q = layer['attention']['q_proj'](x)
                k = layer['attention']['k_proj'](x)
                v = layer['attention']['v_proj'](x)
                
                attn = torch.softmax(q @ k.transpose(-2, -1) / (x.size(-1) ** 0.5), dim=-1)
                attn_out = layer['attention']['out_proj'](attn @ v)
                x = layer['norm1'](x + attn_out)
                
                # Feed forward
                ff_out = layer['feed_forward']['down_proj'](
                    torch.relu(layer['feed_forward']['up_proj'](x))
                )
                x = layer['norm2'](x + ff_out)
            
            # Task-specific heads
            if task_type == "classification":
                return self.classification_head(x.mean(dim=1))  # Pool for classification
            elif task_type == "sequence":
                return self.sequence_head(x[:, 0, :])  # Use first token for sequence classification
            else:  # generation
                return self.generation_head(x)
        
        def prepare_inputs_for_generation(self, input_ids, **kwargs):
            """Required method for PEFT compatibility"""
            return {"input_ids": input_ids}
    
    return MultiTaskTransformer()

def demonstrate_multi_adapter_setup():
    """Show setup of multiple task-specific adapters"""
    
    print("=== Multi-Adapter Setup ===")
    
    # Create base model
    base_model = create_base_model_for_multi_task()
    original_params = sum(p.numel() for p in base_model.parameters())
    print(f"Base model parameters: {original_params:,}")
    
    # Task-specific adapter configurations
    adapter_configs = {
        "sentiment_analysis": LoraConfig(
            r=8,
            lora_alpha=16,
            target_modules=["q_proj", "k_proj", "v_proj"],
            lora_dropout=0.1,
            task_type=TaskType.SEQ_CLS,
        ),
        "text_generation": LoraConfig(
            r=16,
            lora_alpha=32,
            target_modules=["q_proj", "k_proj", "v_proj", "out_proj"],
            lora_dropout=0.05,
            task_type=TaskType.CAUSAL_LM,
        ),
        "sequence_classification": LoraConfig(
            r=4,
            lora_alpha=8,
            target_modules=["q_proj", "v_proj"],
            lora_dropout=0.15,
            task_type=TaskType.SEQ_CLS,
        )
    }
    
    # Apply first adapter, under its own name (get_peft_model names it "default" otherwise)
    peft_model = get_peft_model(base_model, adapter_configs["sentiment_analysis"], adapter_name="sentiment_analysis")
    
    print(f"Initial adapter applied: sentiment_analysis")
    peft_model.print_trainable_parameters()
    
    # Add additional adapters
    for adapter_name, config in list(adapter_configs.items())[1:]:
        peft_model.add_adapter(adapter_name, config)
        print(f"Added adapter: {adapter_name}")
    
    # Show adapter information
    print(f"\nAdapter Management:")
    active_adapters = peft_model.active_adapters
    all_adapters = list(peft_model.peft_config.keys())
    print(f"  All adapters: {all_adapters}")
    print(f"  Active adapters: {active_adapters}")
    assert set(peft_model.peft_config) == set(adapter_configs), all_adapters
    assert active_adapters == ["sentiment_analysis"], active_adapters  # add_adapter does not activate
    
    return peft_model, adapter_configs

def demonstrate_adapter_switching():
    """Show adapter switching and task-specific inference"""
    
    print(f"\n=== Adapter Switching Demo ===")
    
    peft_model, configs = demonstrate_multi_adapter_setup()
    
    # Simulate adapter switching for different tasks
    tasks_and_adapters = [
        ("sentiment_analysis", "Classify sentiment of movie reviews", TaskType.SEQ_CLS),
        ("text_generation", "Generate coherent text continuations", TaskType.CAUSAL_LM),
        ("sequence_classification", "Binary document classification", TaskType.SEQ_CLS),
    ]
    
    for adapter_name, task_description, task_type in tasks_and_adapters:
        print(f"\n--- Task: {adapter_name} ---")
        print(f"Description: {task_description}")
        print(f"Task type: {task_type}")
        
        # Switch to specific adapter
        peft_model.set_adapter(adapter_name)
        current_adapter = peft_model.active_adapters
        print(f"Active adapter: {current_adapter}")
        assert current_adapter == [adapter_name], current_adapter
        
        # Simulate task-specific inference
        with torch.no_grad():
            test_input = torch.randint(0, 1000, (2, 8))  # Batch of 2, seq len 8
            
            if "generation" in adapter_name:
                output = peft_model.get_base_model()(test_input, task_type="generation")
                print(f"Generation output shape: {output.shape}")
            elif "classification" in adapter_name or "sentiment" in adapter_name:
                output = peft_model.get_base_model()(test_input, task_type="classification")
                print(f"Classification output shape: {output.shape}")
            else:
                output = peft_model.get_base_model()(test_input, task_type="sequence")
                print(f"Sequence classification output shape: {output.shape}")
    
    return peft_model

def demonstrate_adapter_composition():
    """Show adapter composition and merging strategies"""
    
    print(f"\n=== Adapter Composition ===")
    
    peft_model = demonstrate_adapter_switching()
    
    # Adapter operation modes
    operations = {
        "single_adapter": "Use one adapter at a time",
        "adapter_merging": "Merge multiple adapters into base model", 
        "adapter_disable": "Temporarily disable all adapters",
        "adapter_enable": "Re-enable specific adapters"
    }
    
    print(f"Adapter operations:")
    for op_name, description in operations.items():
        print(f"  {op_name}: {description}")
    
    # Demonstrate adapter management operations. disable_adapter() is a CONTEXT MANAGER: the
    # adapters are off inside the block and back on when it exits.
    lora_layers = [m for m in peft_model.modules() if isinstance(m, peft.tuners.lora.LoraLayer)]
    test_input = torch.randint(0, 1000, (2, 8))
    peft_model.eval()  # no LoRA dropout, so outputs are comparable
    with torch.no_grad():
        # LoRA B starts at zero; give the active adapter a visible effect. Only the layers it wraps carry
        # it (sequence_classification: q_proj + v_proj in each of the 4 layers)
        active_layers = [layer for layer in lora_layers if peft_model.active_adapter in layer.lora_B]
        assert len(lora_layers) == 16 and len(active_layers) == 8, (len(lora_layers), len(active_layers))
        for layer in active_layers:
            layer.lora_B[peft_model.active_adapter].weight.normal_(std=0.02)
        with_lora = peft_model.get_base_model()(test_input, task_type="classification")
        # Disable all adapters
        with peft_model.disable_adapter():
            assert all(layer.disable_adapters for layer in lora_layers)
            print(f"\n✓ All adapters disabled")
            without_lora = peft_model.get_base_model()(test_input, task_type="classification")
        # ...and leaving the block re-enables them
        assert not any(layer.disable_adapters for layer in lora_layers)
        print(f"✓ Adapters re-enabled")
        assert not torch.allclose(with_lora, without_lora)
        assert torch.allclose(with_lora, peft_model.get_base_model()(test_input, task_type="classification"))
    
    # Show parameter counts in different states: only the active adapter trains
    with_adapters = sum(p.numel() for p in peft_model.parameters() if p.requires_grad)
    print(f"✓ Trainable parameters with adapters: {with_adapters:,}")
    active = peft_model.peft_config[peft_model.active_adapter]  # sequence_classification: r=4, q_proj + v_proj
    assert with_adapters == 4 * active.r * (512 + 512) * len(active.target_modules) == 32_768, with_adapters
    
    # Adapter efficiency analysis
    print(f"\n=== Adapter Efficiency Analysis ===")
    base_params = sum(p.numel() for p in peft_model.get_base_model().parameters())
    
    # Simulate parameter counts for different adapters
    efficiency_stats = {}
    for adapter_name, config in [("sentiment_analysis", configs["sentiment_analysis"]) 
                                for configs in [{"sentiment_analysis": LoraConfig(r=8, lora_alpha=16)}]]:
        # Estimate adapter parameters
        estimated_params = config.r * 2 * 512 * 4  # rough estimate
        efficiency_stats[adapter_name] = {
            "adapter_params": estimated_params,
            "efficiency": f"{100 * estimated_params / base_params:.3f}%",
            "rank": config.r
        }
    
    print(f"Adapter efficiency (base model: {base_params:,} params):")
    for adapter, stats in efficiency_stats.items():
        print(f"  {adapter}: {stats['adapter_params']:,} params ({stats['efficiency']}), rank={stats['rank']}")

# Run multi-adapter demonstrations
print("=== PEFT Multi-Adapter Management ===")

demonstrate_adapter_composition()

print("\nExample 4: Multi-adapter management completed successfully!")

In [ ]:
# Example 5: Saving and Loading PEFT Models
# Demonstrate efficient model serialization and deployment

from peft import PeftModel, LoraConfig, get_peft_model, TaskType
from transformers import AutoModelForCausalLM
import torch
import tempfile
import os

def demonstrate_save_load_workflow():
    """Show complete save/load workflow for PEFT models"""
    
    print("=== PEFT Save/Load Workflow ===")
    
    # Load a small pretrained model
    print("Loading base model...")
    base_model = AutoModelForCausalLM.from_pretrained(
        "gpt2",
        torch_dtype=torch.float32,
    )
    
    # Configure LoRA
    lora_config = LoraConfig(
        r=8,
        lora_alpha=16,
        target_modules=["c_attn"],  # GPT-2 attention projection
        lora_dropout=0.1,
        bias="none",
        task_type=TaskType.CAUSAL_LM,
    )
    
    # Apply PEFT
    peft_model = get_peft_model(base_model, lora_config)
    
    print("\nPEFT model created:")
    peft_model.print_trainable_parameters()
    
    # Change the adapter weights (as training would), so the saved adapter differs from a fresh one
    print("\nPerturbing the adapter weights...")
    for name, param in peft_model.named_parameters():
        if "lora" in name and param.requires_grad:
            param.data += torch.randn_like(param) * 0.01
    adapter_weights = {name: p.detach().clone() for name, p in peft_model.named_parameters() if "lora_" in name}
    # GPT-2 has 12 blocks; each c_attn gains lora_A and lora_B
    assert len(adapter_weights) == 24, len(adapter_weights)
    
    # Save adapter weights only
    with tempfile.TemporaryDirectory() as tmp_dir:
        adapter_path = os.path.join(tmp_dir, "lora_adapter")
        
        print(f"\nSaving adapter to: {adapter_path}")
        peft_model.save_pretrained(adapter_path)
        
        # Check saved files
        saved_files = os.listdir(adapter_path)
        print(f"Saved files: {saved_files}")
        assert "adapter_config.json" in saved_files and "adapter_model.safetensors" in saved_files
        
        # Get file sizes
        total_size = sum(
            os.path.getsize(os.path.join(adapter_path, f)) 
            for f in saved_files
        )
        print(f"Total adapter size: {total_size / 1024:.2f} KB")
        
        # Load adapter into fresh base model
        print("\nLoading adapter into fresh base model...")
        fresh_base = AutoModelForCausalLM.from_pretrained(
            "gpt2",
            torch_dtype=torch.float32,
        )
        
        loaded_model = PeftModel.from_pretrained(fresh_base, adapter_path, torch_device="cpu")
        
        print("Adapter loaded")
        loaded_model.print_trainable_parameters()
        loaded_weights = {name: p.detach() for name, p in loaded_model.named_parameters() if "lora_" in name}
        assert loaded_weights.keys() == adapter_weights.keys()
        assert all(torch.equal(loaded_weights[name], adapter_weights[name]) for name in adapter_weights)
        print(f"Reloaded adapter: {len(loaded_weights)} LoRA tensors identical to the saved ones")
        
        # Test inference
        print("\nTesting inference...")
        test_input = torch.randint(0, 50257, (2, 10))  # GPT-2 vocab size
        
        with torch.no_grad():
            outputs = loaded_model(test_input)
            logits = outputs.logits
        
        print(f"Input shape: {test_input.shape}")
        print(f"Output logits shape: {logits.shape}")
        assert logits.shape == (2, 10, 50257)
        
        return loaded_model

def demonstrate_adapter_merging():
    """Show how to merge adapters for deployment"""
    
    print("\n=== Adapter Merging for Deployment ===")
    
    # Create PEFT model
    base_model = AutoModelForCausalLM.from_pretrained(
        "gpt2",
        torch_dtype=torch.float32,
    )
    
    lora_config = LoraConfig(
        r=16,
        lora_alpha=32,
        target_modules=["c_attn", "c_proj"],
        lora_dropout=0.05,
        bias="none",
        task_type=TaskType.CAUSAL_LM,
    )
    
    peft_model = get_peft_model(base_model, lora_config)
    
    print("Before merging:")
    peft_model.print_trainable_parameters()
    
    # Give the adapter non-zero weights (lora_B starts at zero), and record the adapted output
    with torch.no_grad():
        for name, param in peft_model.named_parameters():
            if "lora_B" in name:
                param.normal_(std=0.02)
    peft_model.eval()
    test_input = torch.randint(0, 50257, (1, 5))
    with torch.no_grad():
        adapted_logits = peft_model(test_input).logits

    # Merge adapter weights into base model
    print("\nMerging adapter weights into base model...")
    merged_model = peft_model.merge_and_unload()
    
    print(f"Merged model type: {type(merged_model).__name__}")
    print(f"Total parameters: {sum(p.numel() for p in merged_model.parameters()):,}")
    
    # The merged model is a plain GPT-2 model with no LoRA layers left, and it computes the same output
    from peft.tuners.lora import LoraLayer
    assert type(merged_model).__name__ == "GPT2LMHeadModel"
    assert not any(isinstance(m, LoraLayer) for m in merged_model.modules())
    assert not any("lora_" in name for name, _ in merged_model.named_parameters())
    with torch.no_grad():
        outputs = merged_model(test_input)
    print(f"Merged model output shape: {outputs.logits.shape}")
    assert outputs.logits.shape == (1, 5, 50257)
    assert torch.allclose(outputs.logits, adapted_logits, atol=1e-3), (outputs.logits - adapted_logits).abs().max()
    
    return merged_model

# Run demonstrations
loaded_model = demonstrate_save_load_workflow()
merged_model = demonstrate_adapter_merging()

print("\nExample 5: PEFT save/load and merging completed successfully!")